# CodAdapt 0.1.0 — avvio rapido per Kaggle

Classificazione binaria e regressione su DataFrame misti, senza imputazione o encoding manuali.
Gli esempi sintetici sono autonomi e separano training, validation per l'early stopping e test.
Il modello è sperimentale: gli esempi mostrano l'API, non risultati di una competizione.

## Installazione dal futuro repository GitHub

Il comando previsto, da eseguire in una cella **solo dopo** la pubblicazione del repository e del tag, è:

```python
!pip install -qq "git+https://github.com/lucalullo/codadapt.git@v0.1.0"
from codadapt import CodAdapt, CodAdaptClassifier, CodAdaptRegressor
model = CodAdapt(random_state=42, verbosity=0)
```

Questa installazione richiede rete abilitata. Il repository/tag non è dichiarato pubblicato:
il comando GitHub è un'istruzione futura non verificata, distinta dalle celle locali eseguibili.


## Installazione offline da wheel

Allega a un dataset Kaggle `codadapt-0.1.0-py3-none-any.whl` e aggiungi il dataset al notebook.
Se NumPy, pandas e scikit-learn compatibili sono già installati:

```python
!pip install -qq --no-deps "/kaggle/input/codadapt-package/codadapt-0.1.0-py3-none-any.whl"
```

Se mancano dipendenze, prepara con rete tutte le wheel transitive compatibili con Python,
sistema e architettura del kernel Kaggle e allegale nella cartella `wheels`. Poi:

```python
!pip install -qq --no-index --find-links="/kaggle/input/codadapt-package/wheels" codadapt==0.1.0
```

I percorsi sono esempi da adattare al nome reale del dataset. `--no-deps` non installa le dipendenze.
Il pacchetto richiede Python 3.10–3.12, NumPy ≥1.24, pandas ≥2.0 e scikit-learn ≥1.3.
Le verifiche reali delle versioni e dell'esecuzione sono riportate in `BENCHMARKS.md`.
Per la verifica locale seguente installa prima la wheel oppure imposta `CODADAPT_WHEEL` al suo percorso.


In [ ]:
import importlib.metadata
import os
import subprocess
import sys

wheel_path = os.environ.get("CODADAPT_WHEEL")
if wheel_path:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", wheel_path])

print("Python:", sys.version.split()[0])
for package in ("codadapt", "numpy", "pandas", "scikit-learn"):
    print(package, importlib.metadata.version(package))
assert importlib.metadata.version("codadapt") == "0.1.0"

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from codadapt import CodAdapt, CodAdaptClassifier, CodAdaptRegressor

assert CodAdapt is CodAdaptClassifier

## 1. Classificazione binaria

Dati numerici con `np.nan`, categorie testuali con `None` e booleani nullable con `pd.NA`.
Il DataFrame entra direttamente nel modello. Lo split è 60% training, 20% validation e 20% test;
la validation esplicita evita un ulteriore split nascosto. Il test viene usato solo per le metriche.


In [ ]:
def make_data(n_samples=900, random_state=42):
    rng = np.random.default_rng(random_state)
    amount = rng.normal(size=n_samples)
    region = rng.choice(["nord", "centro", "sud"], size=n_samples)
    active = rng.choice([True, False], size=n_samples)
    logits = 2.0 * amount + 1.4 * (region == "nord") + 0.8 * active
    target = np.where(rng.random(n_samples) < 1 / (1 + np.exp(-logits)), "sì", "no")
    frame = pd.DataFrame(
        {"importo": amount, "regione": region, "attivo": pd.array(active, dtype="boolean")}
    )
    frame.loc[rng.choice(n_samples, 60, replace=False), "importo"] = np.nan
    frame.loc[rng.choice(n_samples, 40, replace=False), "regione"] = None
    frame.loc[rng.choice(n_samples, 30, replace=False), "attivo"] = pd.NA
    return frame, target

In [ ]:
X, y = make_data()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=43, stratify=y_dev
)
model = CodAdapt(random_state=42, verbosity=0)
model.fit(X_train, y_train, eval_set=(X_valid, y_valid))
probabilities = model.predict_proba(X_test)
positive = (y_test == model.classes_[1]).astype(int)
print(f"ROC-AUC: {roc_auc_score(positive, probabilities[:, 1]):.4f}")
print(f"Log-loss: {log_loss(y_test, probabilities, labels=model.classes_):.4f}")
print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"Iterazioni eseguite: {model.n_iter_}; stato scelto: {model.best_iteration_}")

new_rows = X_test.iloc[:3].copy()
new_rows.loc[:, "regione"] = "categoria_mai_vista"
new_rows.loc[:, "importo"] = np.nan
print("Previsioni con nuove categorie e mancanti:", model.predict(new_rows))

La categoria `categoria_mai_vista` usa un bucket dedicato. I contributi dei bucket senza
supporto nel training sono nulli; un'interazione che li contiene contribuisce zero.
Il vocabolario non si aggiorna durante `predict`. Numeriche fuori dal range osservato
usano i bin estremi, mentre gli infiniti sono rifiutati.

Per l'API davvero minima, con validation interna, basta `model.fit(X_train, y_train)`;
qui abbiamo passato `eval_set` per rendere visibile la separazione dei dati.


## 2. Regressione

Il task è scelto esplicitamente con `CodAdaptRegressor`, anche quando il target è intero.
L'esempio comprende numeriche nullable, stringhe e mancanti. Le metriche sono RMSE e MAE.


In [ ]:
def make_data(n_samples=900, random_state=42):
    rng = np.random.default_rng(random_state)
    age = rng.uniform(18, 80, size=n_samples)
    segment = rng.choice(["base", "plus", "premium"], size=n_samples)
    count = rng.integers(0, 10, size=n_samples)
    target = 0.8 * age + 12 * (segment == "premium") + 2 * count + rng.normal(0, 3, n_samples)
    frame = pd.DataFrame(
        {"età": age, "segmento": segment, "conteggio": pd.array(count, dtype="Int64")}
    )
    frame.loc[rng.choice(n_samples, 50, replace=False), "età"] = np.nan
    frame.loc[rng.choice(n_samples, 30, replace=False), "segmento"] = None
    frame.loc[rng.choice(n_samples, 30, replace=False), "conteggio"] = pd.NA
    return frame, target

In [ ]:
X, y = make_data()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=43)
model = CodAdaptRegressor(random_state=42, verbosity=0)
model.fit(X_train, y_train, eval_set=(X_valid, y_valid))
predictions = model.predict(X_test)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, predictions):.4f}")
print(f"Iterazioni eseguite: {model.n_iter_}; stato scelto: {model.best_iteration_}")

new_rows = X_test.iloc[:3].copy()
new_rows.loc[:, "segmento"] = "categoria_mai_vista"
new_rows.loc[:, "età"] = np.nan
print("Previsioni con nuove categorie e mancanti:", model.predict(new_rows))

## 3. Adattare il notebook a CSV Kaggle

Indica percorsi reali, target, colonna identificativa e task. La funzione seguente esclude
sempre target e identificativo dalle feature, separa una validation e restituisce una submission.
Non esegue alcun accesso a file finché non fornisci i percorsi. La sezione CSV non è verificata
su una competizione reale e non inventa punteggi.

Verifica il formato richiesto dalla competizione: `probabilities=True` produce probabilità
per la seconda classe; altrimenti produce etichette. Per regressione produce valori reali.
Per classi con un significato specifico controlla `model.classes_` prima della submission.


In [ ]:
def fit_kaggle_csv(
    train_csv, test_csv, target, id_column, task="classification", probabilities=False
):
    train = pd.read_csv(train_csv)
    test = pd.read_csv(test_csv)
    if task not in ("classification", "regression"):
        raise ValueError("Scegli classification oppure regression.")
    if target not in train or id_column not in train or id_column not in test:
        raise ValueError("Controlla target e colonna identificativa nei CSV.")
    if target in test:
        raise ValueError("Il CSV di previsione deve essere privo del target.")
    X = train.drop(columns=[target, id_column])
    y = train[target]
    X_submit = test.drop(columns=[id_column])
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if task == "classification" else None
    )
    model = (
        CodAdapt(random_state=42, verbosity=0)
        if task == "classification"
        else CodAdaptRegressor(random_state=42, verbosity=0)
    )
    model.fit(X_train, y_train, eval_set=(X_valid, y_valid))
    if task == "classification":
        valid_proba = model.predict_proba(X_valid)
        print("Classi:", model.classes_)
        print("Validation log-loss:", log_loss(y_valid, valid_proba, labels=model.classes_))
        predictions = (
            model.predict_proba(X_submit)[:, 1] if probabilities else model.predict(X_submit)
        )
    else:
        valid_predictions = model.predict(X_valid)
        print("Validation RMSE:", np.sqrt(mean_squared_error(y_valid, valid_predictions)))
        predictions = model.predict(X_submit)
    return pd.DataFrame({id_column: test[id_column], target: predictions})


TRAIN_CSV = None  # Esempio: "/kaggle/input/nome-competizione/train.csv"
TEST_CSV = None  # Esempio: "/kaggle/input/nome-competizione/test.csv"
TARGET = "target"  # Sostituire con il nome reale.
ID_COLUMN = "id"  # Sostituire con il nome reale.

if TRAIN_CSV is not None and TEST_CSV is not None:
    submission = fit_kaggle_csv(TRAIN_CSV, TEST_CSV, TARGET, ID_COLUMN, task="classification")
    submission.to_csv("submission.csv", index=False)

## Perimetro e verifiche

Sono supportati DataFrame misti e array NumPy numerici 2D, classificazione binaria e
regressione a un target. Sono rifiutati target mancanti, multiclasse, multilabel, sparse,
date non trasformate, infiniti, complessi e oggetti non supportati. Dopo il fit su
DataFrame, passa DataFrame con gli stessi nomi; l'ordine può cambiare.

`verbosity=0` tace l'avanzamento; warning importanti rimangono visibili. La loss di
training è una somma pesata, la loss di validation è non pesata. Non viene eseguito
un refit sulla validation. `n_iter_` e `best_iteration_` distinguono iterazioni eseguite
e stato migliore ripristinato. I dati grezzi del training non sono conservati nel modello.

Le celle degli esempi sintetici sono predisposte per l'esecuzione contro una wheel
locale. Il benchmark disclosure riassume il protocollo e i limiti dei risultati della release.
Le istruzioni GitHub, i percorsi offline esemplificativi e la sezione CSV senza dati reali
non costituiscono prove di installazione remota o di una competizione Kaggle.
